# Task 21 - Membership Duration and Lifecycle Stages

**Goal:** compute how long a customer has been a member (dynamically), normalise it with a log transform, and segment into lifecycle stages using engagement trends.

**Data mapping and an honest limitation**
- Your data has **no customers, no join / signup / registration dates and no repeated time snapshots** (`crawl_month` is a single value). A true membership duration cannot be computed from it.
- The notebook therefore supports two modes (`DURATION_MODE`):
  - **`"date"`** - real mode. Set `DATE_COL` to any join/start-date column (e.g. a merchant start date you obtain later). Duration = `REFERENCE_DATE - date`, recomputed on every run (default reference = today), so it updates dynamically.
  - **`"proxy"`** (default) - **merchant tenure proxy**: `merchant_rating_count` (number of ratings a merchant has accumulated) grows with time in the marketplace, so `log1p` of it is used as a stand-in for how long the seller has been a member. This is a proxy for *maturity*, not a measured duration, and the output says so.
- **Engagement trend** also needs time. Without it, the notebook uses **momentum = engagement percentile minus tenure percentile**: positive means engagement is ahead of what the seller's tenure would predict (rising), negative means it lags (cooling).

**Lifecycle stages:** New -> Growing -> Established -> Mature (momentum >= 0) / Declining (momentum < 0), found with K-means on normalised tenure and engagement and named by centroid order.

**Input:** newest file before this task (Task 17 `engagement_score` is used when present). **Outputs:** `data/processed/womens_clothing_lifecycle_features.csv`, `reports/task_21_lifecycle_stage_profile.csv`, `reports/task_21_tenure_summary.csv`, `reports/task_21_analysis_status.csv`

In [ ]:
import re, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

def find_project_root():
    """Walk up from the notebook folder until a folder containing data/processed is found."""
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "data" / "processed").exists():
            return cand
    return start.parent

ROOT = find_project_root()
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
RAW = ROOT / "data" / "raw" / "womens_clothing.csv"
PROC.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def pick(df, *candidates, required=True):
    """Return the real column name matching any candidate (ignores case, spaces, underscores)."""
    lookup = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if _norm(cand) in lookup:
            return lookup[_norm(cand)]
    if required:
        raise KeyError(f"None of {candidates} found. Columns: {list(df.columns)}")
    return None

def load_first(*paths):
    """Load the first existing CSV, so each task builds on the newest previous output."""
    for p in paths:
        if Path(p).exists():
            d = pd.read_csv(p)
            d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
            print(f"Loaded: {Path(p).name}  shape={d.shape}")
            return d, Path(p)
    raise FileNotFoundError("None of these files exist: " + ", ".join(str(p) for p in paths))

def add_from_raw(df, *names):
    """Borrow columns missing from df (e.g. title, tags) from data/raw, matched on product_id."""
    missing = [n for n in names if pick(df, n, required=False) is None]
    if not missing:
        return df
    raw = pd.read_csv(RAW); raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
    kd, kr = pick(df, "product_id", required=False), pick(raw, "product_id", required=False)
    if kd is None or kr is None:
        raise KeyError(f"Need product_id in both files to borrow {missing} from raw")
    got = [pick(raw, n) for n in missing if pick(raw, n, required=False)]
    if not got:
        raise KeyError(f"{missing} not found in raw file either")
    add = raw.drop_duplicates(kr)[[kr] + got].rename(columns={kr: kd})
    out = df.merge(add, on=kd, how="left")
    print("Borrowed from raw:", got)
    return out

# Newest cumulative dataset first; each task reads the newest file produced BEFORE it.
CHAIN = ["womens_clothing_high_cardinality_encoded.csv", "womens_clothing_engagement_features.csv",
         "womens_clothing_sentiment_features.csv", "womens_clothing_age_segmented.csv",
         "womens_clothing_review_text_features.csv", "womens_clothing_market_basket_features.csv",
         "womens_clothing_product_text_features.csv", "womens_clothing_product_tag_features.csv",
         "womens_clothing_product_segmented.csv", "womens_clothing_final.csv"]

# Tasks 19-21 outputs sit in front of the older chain.
LATE = ["womens_clothing_lifecycle_features.csv", "womens_clothing_tfidf_features.csv", "womens_clothing_channel_interaction_features.csv"]
FULL = LATE + CHAIN

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
DURATION_MODE = "proxy"          # "proxy" or "date"
DATE_COL = None                  # e.g. "merchant_start_date" when DURATION_MODE == "date"
REFERENCE_DATE = pd.Timestamp.today().normalize()      # dynamic: today's date on every run
TENURE_PROXY_COL = "merchant_rating_count"
K_RANGE = range(4, 7)

## 1. Load data

In [ ]:
df, src = load_first(*[PROC / n for n in FULL[1:]])
units_col = pick(df, "units_sold"); rating_col = pick(df, "rating")
mid_col = pick(df, "merchant_id", required=False) or pick(df, "merchant_name", required=False)
eng_col = pick(df, "engagement_score", required=False)
print(f"Products: {len(df):,} | mode: {DURATION_MODE} | reference date: {REFERENCE_DATE.date()}")
if eng_col is None:
    print("engagement_score not found (Task 17 output) -> using percentile of log units sold as the engagement signal")

## 2. Membership duration (dynamic) and log normalisation

In [ ]:
if DURATION_MODE == "date":
    dcol = pick(df, DATE_COL)
    start = pd.to_datetime(df[dcol], errors="coerce")
    df["lc_tenure_days"] = (REFERENCE_DATE - start).dt.days.clip(lower=0)
    tenure_label = f"days since {dcol}"
else:
    pc = pick(df, TENURE_PROXY_COL)
    # merchant-level attribute: take the maximum observed per merchant so every product of a seller shares one value
    df["lc_tenure_days"] = df.groupby(mid_col)[pc].transform("max") if mid_col else df[pc]
    tenure_label = f"proxy: {pc} (ratings accumulated)"
df["lc_tenure_days"] = df["lc_tenure_days"].fillna(df["lc_tenure_days"].median())

df["lc_tenure_log"] = np.log1p(df["lc_tenure_days"])                                      # compresses the long right tail
df["lc_tenure_norm"] = (df["lc_tenure_log"] - df["lc_tenure_log"].min()) / (df["lc_tenure_log"].max() - df["lc_tenure_log"].min() + 1e-9)
df["lc_tenure_pct"] = df["lc_tenure_days"].rank(pct=True)
df["lc_tenure_mode"] = DURATION_MODE
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["lc_tenure_days"], bins=50, ax=ax[0], color="#c44e52"); ax[0].set_title(f"Raw tenure ({tenure_label})")
sns.histplot(df["lc_tenure_log"], bins=50, ax=ax[1], color="#4c72b0"); ax[1].set_title("After log1p")
plt.tight_layout(); plt.show()
print("Skewness before / after log:", round(df["lc_tenure_days"].skew(), 2), "/", round(df["lc_tenure_log"].skew(), 2))

## 3. Engagement trend (momentum)

In [ ]:
eng_raw = df[eng_col] if eng_col else np.log1p(df[units_col])
df["lc_engagement_pct"] = eng_raw.rank(pct=True)
df["lc_momentum"] = df["lc_engagement_pct"] - df["lc_tenure_pct"]       # > 0: engagement ahead of tenure
df["lc_trend"] = pd.cut(df["lc_momentum"], [-1.01, -0.15, 0.15, 1.01], labels=["cooling", "steady", "rising"]).astype(str)
display(df["lc_trend"].value_counts(normalize=True).round(3))
plt.figure(figsize=(7, 5)); sns.scatterplot(data=df.sample(min(len(df), 1500), random_state=1), x="lc_tenure_norm", y="lc_engagement_pct", hue="lc_trend", alpha=.6, palette={"cooling": "#c44e52", "steady": "#999999", "rising": "#55a868"})
plt.title("Tenure vs engagement"); plt.show()

## 4. Lifecycle stages (K-means on tenure and engagement)

In [ ]:
Xl = StandardScaler().fit_transform(df[["lc_tenure_norm", "lc_engagement_pct", "lc_momentum"]])
rows = []
for k in K_RANGE:
    km = KMeans(k, n_init=10, random_state=RANDOM_STATE).fit(Xl)
    rows.append({"k": k, "silhouette": silhouette_score(Xl, km.labels_, sample_size=min(8000, len(Xl)), random_state=RANDOM_STATE)})
kscan = pd.DataFrame(rows); display(kscan.round(4))
best_k = int(kscan.sort_values("silhouette", ascending=False).iloc[0]["k"])
km = KMeans(best_k, n_init=10, random_state=RANDOM_STATE).fit(Xl)
df["_c"] = km.labels_
cent = df.groupby("_c")[["lc_tenure_norm", "lc_engagement_pct", "lc_momentum"]].mean().sort_values("lc_tenure_norm")

def stage_names(cent):
    """Order clusters by tenure; the most mature cluster is split by momentum, the earlier ones are New/Growing/Established."""
    n = len(cent); names = {}
    base = ["New", "Growing", "Established", "Established (late)", "Established (late+)"]
    for i, (c, r) in enumerate(cent.iterrows()):
        if i == n - 1: names[c] = "Mature" if r["lc_momentum"] >= 0 else "Declining"
        elif i < 3: names[c] = base[i]
        else: names[c] = "Established"
    return names
names = stage_names(cent); order = {n: i for i, n in enumerate(["New", "Growing", "Established", "Mature", "Declining"])}
df["lifecycle_stage"] = df["_c"].map(names)
df["lifecycle_stage_id"] = df["lifecycle_stage"].map(order).fillna(2).astype(int)
df = df.drop(columns="_c")
print("k =", best_k); cent.assign(stage=cent.index.map(names)).round(3)

In [ ]:
# Stages sharing a name (e.g. two "Established" clusters) are merged by name; refine rule-based "Declining" within Mature
mature = df["lifecycle_stage"] == "Mature"
df.loc[mature & (df["lc_trend"] == "cooling"), "lifecycle_stage"] = "Declining"
df["lifecycle_stage_id"] = df["lifecycle_stage"].map(order).fillna(2).astype(int)
stage_order = [s for s in ["New", "Growing", "Established", "Mature", "Declining"] if s in df["lifecycle_stage"].unique()]
print(df["lifecycle_stage"].value_counts(normalize=True).reindex(stage_order).round(3))
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
sns.boxplot(data=df, x="lifecycle_stage", y="lc_tenure_days", order=stage_order, ax=ax[0], showfliers=False); ax[0].set_title("Tenure by stage"); ax[0].set_yscale("log")
sns.boxplot(data=df, x="lifecycle_stage", y="lc_engagement_pct", order=stage_order, ax=ax[1]); ax[1].set_title("Engagement percentile by stage")
sns.countplot(data=df, x="lifecycle_stage", hue="lc_trend", order=stage_order, ax=ax[2]); ax[2].set_title("Trend mix by stage")
plt.tight_layout(); plt.show()

In [ ]:
g = df.groupby("lifecycle_stage")
profile = g.agg(products=(rating_col, "size"), median_tenure=("lc_tenure_days", "median"), mean_tenure_log=("lc_tenure_log", "mean"),
                mean_engagement_pct=("lc_engagement_pct", "mean"), mean_momentum=("lc_momentum", "mean"),
                median_units=(units_col, "median"), mean_rating=(rating_col, "mean")).reindex(stage_order).reset_index()
profile["share"] = profile["products"] / profile["products"].sum(); profile.insert(1, "stage_id", profile["lifecycle_stage"].map(order))
profile.round(3)

## 5. Save outputs

In [ ]:
out = df.copy()
out_path = PROC / "womens_clothing_lifecycle_features.csv"; out.to_csv(out_path, index=False)
profile.round(5).to_csv(REPORTS / "task_21_lifecycle_stage_profile.csv", index=False)
df["lc_tenure_days"].describe().rename("lc_tenure_days").to_frame().join(df["lc_tenure_log"].describe().rename("lc_tenure_log")).round(4).reset_index().rename(columns={"index": "statistic"}).to_csv(REPORTS / "task_21_tenure_summary.csv", index=False)
base_cols = pd.read_csv(src, nrows=1).columns
status = pd.DataFrame([{"task": "Task 21 - Membership Duration & Lifecycle", "status": "completed", "run_at": datetime.now().isoformat(timespec="seconds"),
    "input_file": src.name, "rows": len(out), "duration_mode": DURATION_MODE, "tenure_source": tenure_label, "reference_date": str(REFERENCE_DATE.date()),
    "kmeans_k": best_k, "n_stages": len(stage_order), "skew_before_log": round(df["lc_tenure_days"].skew(), 3), "skew_after_log": round(df["lc_tenure_log"].skew(), 3),
    "columns_added": len([c for c in out.columns if c not in base_cols]), "note": "proxy for tenure, not measured membership" if DURATION_MODE == "proxy" else "",
    "output_dataset": out_path.name}])
status.to_csv(REPORTS / "task_21_analysis_status.csv", index=False); status.T

## 6. Key takeaways (fill in after running)

- How much did the log transform reduce skew, and are the stage boundaries sensible on the log scale?
- Which stage holds the most products, and do Declining sellers show lower ratings or sales than Mature ones?
- In `proxy` mode describe results as **seller maturity**, not membership length; once you have a real start-date column, set `DURATION_MODE = "date"` and `DATE_COL` and rerun.
- Momentum is a cross-sectional proxy for trend. A true engagement trend needs repeated snapshots of the same sellers over time.